# 04 — 30-Day Landmark Sensitivity Analysis

Synthetic demonstration of the 30-day landmark sensitivity analysis used to evaluate whether early mortality materially influences the estimated association between treatment and overall survival. The analysis is restricted to patients who survive and remain under observation beyond the prespecified 30-day landmark, with follow-up re-originated at the landmark.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..")
df = pd.read_csv(ROOT / "data" / "synthetic_dlbcl_demo.csv")
df.head()


In [ ]:
from statsmodels.duration.hazard_regression import PHReg

m = df.copy()
m["CIT"] = (m["treatment"]=="Chemoimmunotherapy").astype(int)
m["male"] = (m["sex"]=="Male").astype(int)
m["stage_III"] = (m["ann_arbor_stage"]=="III").astype(int)
m["stage_IV"] = (m["ann_arbor_stage"]=="IV").astype(int)

landmark_months = 1.0
lm = m[m["survival_months"] > landmark_months].copy()
lm["survival_from_landmark"] = lm["survival_months"] - landmark_months

X = lm[["CIT","age","male","stage_III","stage_IV","comorbidity_ge1","b_symptoms"]].astype(float)
model = PHReg(lm["survival_from_landmark"], X, status=lm["death"])
result = model.fit()
print(result.summary())
